## Data generation

### Import and setup

In [57]:
# Importing necessary libraries

import numpy as np
import pandas as pd

In [58]:
# Set a random seed for reproducible results

np.random.seed(42)

### Simulating Funnel Drop-offs & Submissions

In [59]:
# Defining sample sizes for a balanced A/B test

n_A = 10000
n_B = 10000

total_n = n_A + n_B

print(f"Simulating data for {total_n} total users (A: {n_A}, B: {n_B})")

Simulating data for 20000 total users (A: 10000, B: 10000)


In [60]:
# Creating base DataFrame

df_A = pd.DataFrame({'group': ['A'] * n_A})
df_B = pd.DataFrame({'group': ['B'] * n_B})

In [61]:
# Submission rates: Group B is expected to have a lower submission rate due to the additional friction of Step 0

target_sub_rate_effect = 0.08 # Target effect on submission rate for Group B compared to Group A

target_sub_rate_A = 0.82  # Target submission rate for Group A
target_sub_rate_B = target_sub_rate_A - target_sub_rate_effect  # Target submission rate for Group B

In [62]:
print(target_sub_rate_A, target_sub_rate_B)

0.82 0.74


In [63]:
p_A1_to_A2 = 0.95  # Probability of progressing from Step 1 to Step 2 in Group A
p_A2_to_A3 = 0.925  # Probability of progressing from Step 2 to Step 3 in Group A
p_A3_to_Asub = target_sub_rate_A / (p_A1_to_A2 * p_A2_to_A3)  # Probability of progressing from Step 3 to submission in Group A

In [64]:
p_B0_to_B1 = 0.90  # Probability of progressing from Step 0 to Step 1 in Group B
p_B1_to_B2 = p_A1_to_A2 #- np.random.uniform(0.005, 0.010)#  # Probability of progressing from Step 1 to Step 2 in Group B (slightly lower than Group A)
p_B2_to_B3 = p_A2_to_A3 #- np.random.uniform(0.005, 0.010)#  # Probability of progressing from Step 2 to Step 3 in Group B (slightly lower than Group A)
p_B3_to_Bsub = target_sub_rate_B / (p_B0_to_B1 * p_B1_to_B2 * p_B2_to_B3) #- np.random.uniform(0.005, 0.010)# # Probability of progressing from Step 3 to submission in Group B (slightly lower than Group A)

In [65]:
print(p_A1_to_A2, p_A2_to_A3, p_A3_to_Asub)
print(p_B0_to_B1, p_B1_to_B2, p_B2_to_B3, p_B3_to_Bsub)

0.95 0.925 0.9331436699857751
0.9 0.95 0.925 0.935672514619883


In [66]:
# Version A Funnel Logic

df_A['reached_step_1'] = 1  # Starts at Step 1
df_A['reached_step_2'] = np.random.binomial(1, p_A1_to_A2, n_A)
df_A['reached_step_3'] = df_A['reached_step_2'] * np.random.binomial(1, p_A2_to_A3, n_A)
df_A['submitted'] = df_A['reached_step_3'] * np.random.binomial(1, p_A3_to_Asub, n_A)

In [67]:
# Version B Funnel Logic

df_B['reached_step_0'] = 1  # Starts at Step 0
df_B['reached_step_1'] = np.random.binomial(1, p_B0_to_B1, n_B) # Drop-off due to disclaimer agreement
df_B['reached_step_2'] = df_B['reached_step_1'] * np.random.binomial(1, p_B1_to_B2, n_B)
df_B['reached_step_3'] = df_B['reached_step_2'] * np.random.binomial(1, p_B2_to_B3, n_B)
df_B['submitted'] = df_B['reached_step_3'] * np.random.binomial(1, p_B3_to_Bsub, n_B)

In [68]:
# Combining the datasets

df = pd.concat([df_A, df_B], ignore_index = True).astype({'reached_step_0': float, 'reached_step_1': float, 'reached_step_2': float, 'reached_step_3': float, 'submitted': float})[['group', 'reached_step_0', 'reached_step_1', 'reached_step_2', 'reached_step_3', 'submitted']]    

### Simulating Estimated Costs & Hotline Calls

In [86]:
# Cost difference: Group B is expected to have a lower estimated cost due to the additional friction of Step 0

target_cost_diff = 1200  # Target difference in estimated costs between Group B and Group A

target_cost_A = 4500  # Estimated cost for Group A
target_cost_B = target_cost_A - target_cost_diff  # Estimated cost for Group B

In [ ]:
sigma = 0.6  # Standard deviation for the log-normal distribution

mu_A = np.log(target_cost_A) - (sigma**2 / 2) # Mean for the log-normal distribution of Group A costs

In [ ]:
df['estimated_cost'] = np.random.lognormal(mean = mu_A, sigma = sigma, size = total_n).round(0)

df.loc[df['group'] == 'B', 'estimated_cost'] -= target_cost_diff

df.loc[df['reached_step_2'] == 0, 'estimated_cost'] = np.nan

In [72]:
# Hotline Calls

df['hotline_calls'] = np.random.poisson(lam = 0.15, size = total_n)

### KPI Aggregation and Validation

In [83]:
# 1. Submission Rate

print("=== FUNNEL PERFORMANCE & DROP-OFFS ===")

funnel_metrics = df.groupby('group')[['reached_step_0', 'reached_step_1', 'reached_step_2', 'reached_step_3', 'submitted']].mean()

print(funnel_metrics.round(4).to_string(formatters={  
        'reached_step_0': '{:.2%}'.format,
        'reached_step_1': '{:.2%}'.format,
        'reached_step_2': '{:.2%}'.format,
        'reached_step_3': '{:.2%}'.format,
        'submitted': '{:.2%}'.format
}))

sub_rate_A = funnel_metrics.loc['A', 'submitted']
sub_rate_B = funnel_metrics.loc['B', 'submitted']

sub_rate_effect = sub_rate_B - sub_rate_A

print(f"\nObserved Submission Rate Effect: {sub_rate_effect:.2%}")

=== FUNNEL PERFORMANCE & DROP-OFFS ===
      reached_step_0 reached_step_1 reached_step_2 reached_step_3 submitted
group                                                                      
A                NaN        100.00%         95.26%         87.87%    81.95%
B            100.00%         89.86%         85.61%         79.45%    74.68%

Observed Submission Rate Effect: -7.27%


In [74]:
# 2. Drop-off Analysis

print("\n=== GRANULAR STEP-TO-STEP FRICTION ANALYSIS ===")

sums = df.groupby('group')[['reached_step_0', 'reached_step_1', 'reached_step_2', 'reached_step_3', 'submitted']].sum()

# 1. Compute conditional drop-offs (as you did)
drop_0_to_1 = 1 - (sums['reached_step_1'] / sums['reached_step_0'])
drop_1_to_2 = 1 - (sums['reached_step_2'] / sums['reached_step_1'])
drop_2_to_3 = 1 - (sums['reached_step_3'] / sums['reached_step_2'])
drop_3_to_sub = 1 - (sums['submitted'] / sums['reached_step_3'])

# 2. Build an explicit step-level comparison matrix
# This handles the structural mismatch of Step 0 cleanly
steps_labels = ['Step 0 -> Step 1', 'Step 1 -> Step 2', 'Step 2 -> Step 3', 'Step 3 -> Submit']

drop_rates_A = [np.nan, drop_1_to_2['A'], drop_2_to_3['A'], drop_3_to_sub['A']]
drop_rates_B = [drop_0_to_1['B'], drop_1_to_2['B'], drop_2_to_3['B'], drop_3_to_sub['B']]

# 3. Create a clean pandas DataFrame for step evaluation
step_evaluation = pd.DataFrame({
    'Drop Rate A': drop_rates_A,
    'Drop Rate B': drop_rates_B
}, index = steps_labels)

# 4. Calculate Absolute Difference (Treatment - Control)
# Positive means Group B experienced MORE friction/drop-off at this specific step
step_evaluation['Absolute Delta (B - A)'] = step_evaluation['Drop Rate B'] - step_evaluation['Drop Rate A']

# 5. Calculate Relative Friction Ratio (B / A)
# Measures the proportional increase in friction (e.g., 1.10x means 10% more friction in B)
step_evaluation['Friction Lift Ratio (B/A)'] = step_evaluation['Drop Rate B'] / step_evaluation['Drop Rate A']

# Display the matrix formatted beautifully
print(step_evaluation.round(4).to_string(formatters={
    'Drop Rate A': '{:.2%}'.format,
    'Drop Rate B': '{:.2%}'.format,
    'Absolute Delta (B - A)': '{:+.2%}'.format,
    'Friction Lift Ratio (B/A)': '{:.2f}x'.format
}))


=== GRANULAR STEP-TO-STEP FRICTION ANALYSIS ===
                 Drop Rate A Drop Rate B Absolute Delta (B - A) Friction Lift Ratio (B/A)
Step 0 -> Step 1         NaN      10.14%                    NaN                       NaN
Step 1 -> Step 2       4.74%       4.73%                 -0.01%                     1.00x
Step 2 -> Step 3       7.76%       7.20%                 -0.56%                     0.93x
Step 3 -> Submit       6.74%       6.00%                 -0.73%                     0.89x


In [91]:
# 3. Estimated Costs Analysis

print("=== ESTIMATED CLAIMS COSTS ===")

cost_summary = df.groupby('group')['estimated_cost'].agg(['mean', 'median'])

print(cost_summary.round(2))

mean_diff_costs = cost_summary.loc['B', 'mean'] - cost_summary.loc['A', 'mean']

print(f"Mean Cost Difference: {mean_diff_costs:.1f} EUR")

=== ESTIMATED CLAIMS COSTS ===
          mean  median
group                 
A      4531.91  3749.5
B      3288.55  2748.0
Mean Cost Difference: -1243.4 EUR


In [82]:
# 4. Hotline Calls Analysis

print("=== HOTLINE CALLS ===")

hotline_summary = df.groupby('group')['hotline_calls'].agg(['mean', 'median'])

print(hotline_summary.round(4))

mean_diff_hotline = hotline_summary.loc['B', 'mean'] - hotline_summary.loc['A', 'mean']

print(f"Mean Hotline Calls Difference: {mean_diff_hotline:.1f}")

=== HOTLINE CALLS ===
         mean  median
group                
A      0.1451     0.0
B      0.1487     0.0
Mean Hotline Calls Difference: 0.0


### The dataset

In [80]:
df

,group,reached_step_0,reached_step_1,reached_step_2,reached_step_3,submitted,estimated_cost,hotline_calls
0,A,NaN,1.0,1.0,1.0,1.0,2272.0,0
1,A,NaN,1.0,0.0,0.0,0.0,NaN,0
2,A,NaN,1.0,1.0,1.0,1.0,2096.0,0
3,A,NaN,1.0,1.0,1.0,1.0,5303.0,1
4,A,NaN,1.0,1.0,1.0,1.0,2849.0,0
...,...,...,...,...,...,...,...,...
19995,B,1.0,1.0,1.0,0.0,0.0,3053.0,0
19996,B,1.0,1.0,1.0,1.0,0.0,1758.0,0
19997,B,1.0,1.0,1.0,1.0,1.0,1910.0,0
19998,B,1.0,1.0,1.0,1.0,1.0,1202.0,0
